# AMMS 302 — Week 9: สอบกลางภาค (Midterm) — Review & Practice
**ทบทวนสัปดาห์ 1–8: pandas · ETL · SQLite DDL/DML · Queries · JOIN/GROUP BY/HAVING**

> เปิดคู่กับ [สไลด์ wk09](./wk09.html) — โน้ตบุ๊กนี้คือแบบฝึกหัดจำลอง (mock exam) ให้ลองทำเองก่อนแล้วดูเฉลยท้ายข้อ

---


In [ ]:
# Setup — healthinfo.db + prescriptions/referrals จาก week07 (ถ้าไม่มี รัน week05/07 notebook ก่อน)
import sqlite3, pathlib, pandas as pd
db = pathlib.Path("healthinfo.db")
assert db.exists(), "run week05-sql-basics.ipynb first"
con = sqlite3.connect(db); cur = con.cursor()
cur.execute("PRAGMA foreign_keys = ON")
tables = [r[0] for r in cur.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
print("tables:", tables)

# ถ้ายังไม่มี prescriptions (จาก week07) ให้ seed แบบย่อ
if 'prescriptions' not in tables:
    cur.execute("CREATE TABLE prescriptions(rx_id INTEGER PRIMARY KEY, patient_id INTEGER NOT NULL REFERENCES patients(patient_id), drug TEXT NOT NULL, dose_mg REAL, rx_date TEXT, cost_thb REAL)")
    cur.executemany("INSERT OR IGNORE INTO prescriptions VALUES (?,?,?,?,?,?)", [
     (1,10001,'Metformin',500,'2025-03-01',35.0),(2,10001,'Lisinopril',10,'2025-03-01',42.0),
     (3,10002,'Metformin',850,'2025-03-02',55.0),(4,10003,'Atorvastatin',20,'2025-03-03',68.0),
     (5,10003,'Metformin',500,'2025-04-01',35.0)])
    con.commit(); print("seeded prescriptions")
for t in ['patients','prescriptions']:
    print(t, cur.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0], "rows")

## 📝 Part A — SQL Written (10 ข้อ × 3 คะแนน)
*เขียน query ในเซลล์ว่างใต้แต่ละข้อ แล้วรันเทียบเฉลย*

**A1.** นับผู้ป่วยทั้งหมดในตาราง patients  
**A2.** หญิงที่มี HbA1c > 7 แสดง hn, hba1c เรียง a1c มาก→น้อย  
**A3.** ผู้ป่วยที่ยังไม่ได้วัด HbA1c (NULL)  
**A4.** ค่าเฉลี่ย/ต่ำสุด/สูงสุดของ hba1c (1 แถว)  
**A5.** จำนวนเพศที่ไม่ซ้ำ (DISTINCT)  
**A6.** hn ที่ขึ้นต้น 'HN-54' 5 รายการ  
**A7.** ผู้ป่วยอายุ (จาก birth_date) ≥ 60 ปี ใช้ julianday  
**A8.** INNER JOIN patients×prescriptions แสดง hn, drug, cost  
**A9.** ยอด cost รวมต่อชนิดยา (GROUP BY drug) เรียงมาก→น้อย  
**A10.** เฉพาะยาที่ถูกจ่าย > 1 ครั้ง (HAVING)


In [ ]:
# ✍️ A1–A10: เขียน query ของคุณที่นี่ (แยกเป็นหลาย pd.read_sql ก็ได้)



<details><summary>🔑 เฉลย Part A (กด expand)</summary>

```sql
-- A1
SELECT COUNT(*) FROM patients;
-- A2
SELECT hn, hba1c FROM patients WHERE gender='หญิง' AND hba1c>7 ORDER BY hba1c DESC;
-- A3
SELECT hn FROM patients WHERE hba1c IS NULL;          -- = NULL ❌
-- A4
SELECT AVG(hba1c), MIN(hba1c), MAX(hba1c) FROM patients;
-- A5
SELECT COUNT(DISTINCT gender) FROM patients;
-- A6
SELECT hn FROM patients WHERE hn LIKE 'HN-54%' LIMIT 5;
-- A7
SELECT hn, CAST((julianday('now')-julianday(birth_date))/365.25 AS INT) age
FROM patients WHERE birth_date LIKE '____-__-__'
  AND (julianday('now')-julianday(birth_date))/365.25 >= 60;
-- A8
SELECT p.hn, r.drug, r.cost_thb FROM patients p
JOIN prescriptions r ON r.patient_id=p.patient_id;
-- A9
SELECT drug, SUM(cost_thb) total FROM prescriptions GROUP BY drug ORDER BY total DESC;
-- A10
SELECT drug, COUNT(*) n FROM prescriptions GROUP BY drug HAVING COUNT(*)>1;
```
</details>


## 📝 Part B — pandas Hands-on (2 ข้อ × 10 คะแนน)
**B1.** อ่าน `patients_data.csv` → ทำความสะอาด: dropna `HbA1c_level` + กรองอายุ 0–89 + บันทึก `clean_midterm.csv`  
**B2.** จาก B1: groupby `gender` สรุป mean bp / count — เทียบกับ SQL ใน A-part


In [ ]:
# ✍️ B1–B2



<details><summary>🔑 เฉลย Part B</summary>

```python
# B1
df = pd.read_csv('patients_data.csv')
df = df.dropna(subset=['HbA1c_level'])
df = df[(df['Age']>=0)&(df['Age']<=89)]
df.to_csv('clean_midterm.csv', index=False)
# B2
print(df.groupby('gender').agg(mean_bp=('systolic_bp','mean'), n=('subject_id','count')))
# เทียบ SQL:
print(pd.read_sql("SELECT gender, AVG(systolic_bp) mbp, COUNT(*) n FROM patients GROUP BY gender", con))
```
</details>


## 📝 Part C — Concept MCQ-style (ตอบใน comment)
C1. `WHERE hba1c = NULL` ได้กี่แถว? ทำไม?  
C2. INNER vs LEFT JOIN ต่างกันอย่างไร (1 ประโยค)?  
C3. WHERE กับ HAVING ใครทำงานก่อน?  
C4. Schema-on-Write คือ Data ___ ?  
C5. 18 identifiers ของ HIPAA Safe Harbor เกี่ยวกับสัปดาห์ไหน?


<details><summary>🔑 เฉลย C</summary>

- **C1:** 0 แถว — NULL เทียบได้ UNKNOWN (three-valued logic) ต้องใช้ `IS NULL`
- **C2:** INNER เอาเฉพาะ match, LEFT รักษาตารางซ้ายครบ (ไม่ match → NULL)
- **C3:** WHERE (rows) ก่อน GROUP BY, HAVING (groups) หลัง
- **C4:** Data Warehouse (Data Lake = Schema-on-Read)
- **C5:** Week 10 — de-identification/HIPAA Safe Harbor
</details>


In [ ]:
# Final check + close
con.commit()
print("tables:", [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")])
con.close()
print("พร้อมสอบ! — นำ healthinfo.db + .ipynb เข้าห้องสอบ (ถ้าอาจารย์อนุญาต)")